# Portable section to make it compatible with local and colab

In [23]:
import os
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
    except ImportError:
        print("google.colab unavailable in this runtime; falling back to local paths.")
        IN_COLAB = False

PROJECT_ROOT = "/content/drive/MyDrive/TheRealDNNExp" if IN_COLAB else os.path.abspath("..")
MODULES_DIR = os.path.join(PROJECT_ROOT, "Problem2", "modules")
RESULTS_CSV_PATH = os.path.join(PROJECT_ROOT, "Results", "results.csv")
FIGURES_DIR = os.path.join(PROJECT_ROOT, "Problem2", "figures")

if MODULES_DIR not in sys.path:
    sys.path.insert(0, MODULES_DIR)

print(f"Project root: {PROJECT_ROOT}")
print(f"Modules dir: {MODULES_DIR}")
print(f"Results CSV path: {RESULTS_CSV_PATH}")
print(f"Figures dir: {FIGURES_DIR}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Project root: /content/drive/MyDrive/TheRealDNNExp
Modules dir: /content/drive/MyDrive/TheRealDNNExp/Problem2/modules
Results CSV path: /content/drive/MyDrive/TheRealDNNExp/Results/results.csv
Figures dir: /content/drive/MyDrive/TheRealDNNExp/Problem2/figures


In [24]:
# short cuts to load my new .py modlues and few ROOT path w.o. re start the sessions or to survive session restart
import os
import pandas as pd
import sys

# Leaf-to-root reload order — safe to run blindly, any time
_P2_MODULE_ORDER = [
    "attention",
    "positional_encoding",
    "results_io",
    "multihead_attention", # depends on attention
    "encoder",             # depends on multihead_attention
    "har_encoder",         # depends on encoder, positional_encoding
    "char_lm",             # depends on attention, encoder, positional_encoding
    "analysis",            # depends on attention
]

for _mod_name in _P2_MODULE_ORDER:
    if _mod_name in sys.modules:
        importlib.reload(sys.modules[_mod_name])

from attention import *
from positional_encoding import *
from results_io import *
from multihead_attention import *
from encoder import *
from har_encoder import *
from char_lm import *
from analysis import *

# Problem 2: Transformer from Scratch
Student: Nilesh Ranjan Mitra | ID: 2026aj05496
d=128 | heads=4 | layers=4 | Seeds: 42, 123, 999

## Step 2 — Attention from Scratch (P2.1)
Scaled dot-product attention with optional mask, verified against
torch.nn.functional.scaled_dot_product_attention, plus a causal-mask
test and a hand-computed 3-token example (no instructor example given,
per Prof's second instruction — values chosen here).

In [25]:
# Unit tests: softmax rows sum to 1, matches PyTorch, causal mask
import torch
import torch.nn.functional as F
from attention import scaled_dot_product_attention, causal_mask

torch.manual_seed(0)
B, H, T, D_K = 2, 1, 5, 8
q = torch.randn(B, H, T, D_K)
k = torch.randn(B, H, T, D_K)
v = torch.randn(B, H, T, D_K)

# Test 1: each row of weights sums to 1
_, weights = scaled_dot_product_attention(q, k, v, return_weights=True)
row_sums = weights.sum(dim=-1)
assert torch.allclose(row_sums, torch.ones_like(row_sums), atol=1e-6), "attention rows must sum to 1"

# Test 2: matches PyTorch's reference implementation within 1e-5
our_out = scaled_dot_product_attention(q, k, v)
ref_out = F.scaled_dot_product_attention(q, k, v)
max_err = (our_out - ref_out).abs().max().item()
assert max_err < 1e-5, f"mismatch vs PyTorch: {max_err}"
print(f"Max abs error vs F.scaled_dot_product_attention: {max_err:.2e}")

# Test 3: causal mask gives zero weight to future positions
mask = causal_mask(T)
_, causal_weights = scaled_dot_product_attention(q, k, v, mask=mask, return_weights=True)
upper_triangle = causal_weights.triu(diagonal=1)
assert torch.allclose(upper_triangle, torch.zeros_like(upper_triangle), atol=1e-6), "future positions must have zero weight"

# Causal version should also match PyTorch's own causal path
ref_causal_out = F.scaled_dot_product_attention(q, k, v, is_causal=True)
our_causal_out = scaled_dot_product_attention(q, k, v, mask=mask)
causal_err = (our_causal_out - ref_causal_out).abs().max().item()
assert causal_err < 1e-5, f"causal mismatch vs PyTorch: {causal_err}"
print(f"Max abs error (causal) vs PyTorch: {causal_err:.2e}")

print("All attention unit tests passed.")

Max abs error vs F.scaled_dot_product_attention: 1.19e-07
Max abs error (causal) vs PyTorch: 1.19e-07
All attention unit tests passed.


### Hand-computed 3-token example (d_k = 2)

Q = K =
[[1, 0],
 [0, 1],
 [1, 1]]

V =
[[1,   0  ],
 [0,   1  ],
 [0.5, 0.5]]

Scale = 1 / sqrt(2) = 0.7071

**Row 1** (q1=[1,0]): scores = [1, 0, 1] * 0.7071 = [0.7071, 0, 0.7071]
softmax -> [0.4011, 0.1978, 0.4011]
output = 0.4011*[1,0] + 0.1978*[0,1] + 0.4011*[0.5,0.5] = **[0.6017, 0.3983]**

**Row 2** (q2=[0,1]): scores = [0, 1, 1] * 0.7071 = [0, 0.7071, 0.7071]
softmax -> [0.1978, 0.4011, 0.4011]
output = **[0.3983, 0.6017]** (by symmetry with row 1)

**Row 3** (q3=[1,1]): scores = [1, 1, 2] * 0.7071 = [0.7071, 0.7071, 1.4142]
softmax -> [0.2482, 0.2482, 0.5035]
output = 0.2482*[1,0] + 0.2482*[0,1] + 0.5035*[0.5,0.5] ≈ **[0.5000, 0.5000]**

In [26]:
# Verifying the hand calculation in code
Q = torch.tensor([[1.0, 0.0], [0.0, 1.0], [1.0, 1.0]])
K = Q.clone()
V = torch.tensor([[1.0, 0.0], [0.0, 1.0], [0.5, 0.5]])

hand_output = torch.tensor([
    [0.6017, 0.3983],
    [0.3983, 0.6017],
    [0.5000, 0.5000],
])

code_output = scaled_dot_product_attention(Q, K, V)
max_err = (code_output - hand_output).abs().max().item()
assert max_err < 1e-3, f"hand vs code mismatch: {max_err}"
print("Code output:\n", code_output)
print(f"Max abs error vs hand calculation: {max_err:.2e}")
print("Hand calculation and code agree.")

Code output:
 tensor([[0.6017, 0.3983],
        [0.3983, 0.6017],
        [0.5000, 0.5000]])
Max abs error vs hand calculation: 3.19e-05
Hand calculation and code agree.


## Step 3 — Multi-Head Attention and Pre-LN Encoder Block (P2.2)
W_Q/W_K/W_V/W_O with head reshaping (built on the Task 1 attention core),
a pre-LayerNorm encoder block with a 4d-wide feed-forward layer, residual
connections and dropout, and sinusoidal/learned/none positional encodings.
Parameter count is derived by hand as 4d^2 + 2*d*d_ff per block and checked
against sum(p.numel()).

In [27]:
# import + shape test for MultiHeadAttention:

from multihead_attention import MultiHeadAttention

torch.manual_seed(0)
D_MODEL, N_HEADS, SEQ_LEN, BATCH = 128, 4, 7, 3
mha = MultiHeadAttention(d_model=D_MODEL, n_heads=N_HEADS)
x = torch.randn(BATCH, SEQ_LEN, D_MODEL)

# Shape must be preserved: (B, T, d_model) -> (B, T, d_model)
out = mha(x)
assert out.shape == (BATCH, SEQ_LEN, D_MODEL), f"unexpected shape {out.shape}"

# Causal masking inside MHA: with a causal mask, changing a future token
# must not change an earlier position's output.
from attention import causal_mask
mask = causal_mask(SEQ_LEN)
out_causal_1 = mha(x, mask=mask)
x_perturbed = x.clone()
x_perturbed[:, -1, :] += 10.0  # perturb only the LAST position
out_causal_2 = mha(x_perturbed, mask=mask)
assert torch.allclose(out_causal_1[:, :-1, :], out_causal_2[:, :-1, :], atol=1e-5), \
    "causal MHA: earlier positions must be unaffected by a later-position perturbation"

print("MultiHeadAttention shape + causal-independence tests passed.")

MultiHeadAttention shape + causal-independence tests passed.


In [28]:
# import + shape test for positional encodings:

from positional_encoding import (
    SinusoidalPositionalEncoding, LearnedPositionalEncoding, NoPositionalEncoding
)

pe_sin = SinusoidalPositionalEncoding(d_model=D_MODEL)
pe_learned = LearnedPositionalEncoding(d_model=D_MODEL)
pe_none = NoPositionalEncoding()

x = torch.zeros(BATCH, SEQ_LEN, D_MODEL)

# "none" must be a true identity
assert torch.equal(pe_none(x), x), "NoPositionalEncoding must be identity"

# sinusoidal PE values must lie in [-1, 1] (sums of sin/cos)
out_sin = pe_sin(x)
assert out_sin.abs().max() <= 1.0 + 1e-6, "sinusoidal PE values should be bounded by 1"

# sinusoidal PE must differ across positions (so the model can tell them apart)
assert not torch.allclose(out_sin[:, 0, :], out_sin[:, 1, :]), \
    "adjacent positions must have distinct encodings"

# learned PE has trainable parameters; sinusoidal/none do not
assert sum(p.numel() for p in pe_learned.parameters()) == 5000 * D_MODEL
assert sum(p.numel() for p in pe_sin.parameters()) == 0
assert sum(p.numel() for p in pe_none.parameters()) == 0

print("Positional encoding tests passed (none=identity, sinusoidal bounded+distinct, learned has params).")

Positional encoding tests passed (none=identity, sinusoidal bounded+distinct, learned has params).


In [29]:
#  import + shape/residual test for EncoderBlock:

from encoder import EncoderBlock, TransformerEncoder

D_FF = 512
block = EncoderBlock(d_model=D_MODEL, n_heads=N_HEADS, d_ff=D_FF, dropout=0.0)
block.eval()  # dropout off for a deterministic shape/residual check

x = torch.randn(BATCH, SEQ_LEN, D_MODEL)
out = block(x)
assert out.shape == x.shape, f"encoder block must preserve shape, got {out.shape}"

# Residual sanity: zeroing out the MHA+FFN weights should make the block an identity
# (since Pre-LN output = x + sublayer(LN(x)), and sublayer(anything)=0 if W_O/last-linear=0)
with torch.no_grad():
    block.mha.W_O.weight.zero_(); block.mha.W_O.bias.zero_()
    block.ff[-1].weight.zero_(); block.ff[-1].bias.zero_()
identity_out = block(x)
assert torch.allclose(identity_out, x, atol=1e-5), \
    "with zeroed output projections, pre-LN block must reduce to identity (residual-only)"

print("EncoderBlock shape + residual-identity tests passed.")

EncoderBlock shape + residual-identity tests passed.


In [30]:
# parameter-count derivation vs sum(p.numel()):
# Hand derivation for d=128, d_ff=512 (one EncoderBlock):
d, d_ff = 128, 512

weights_only = 4 * d**2 + 2 * d * d_ff          # W_Q,W_K,W_V,W_O + FFN linear1+linear2 weights
mha_biases = 4 * d                              # b_Q, b_K, b_V, b_O
ffn_biases = d_ff + d                           # linear1 bias (d_ff,) + linear2 bias (d,)
ln_params = 2 * (2 * d)                         # 2 LayerNorms x (weight + bias), each (d,)

derived_total = weights_only + mha_biases + ffn_biases + ln_params

fresh_block = EncoderBlock(d_model=d, n_heads=4, d_ff=d_ff)
actual_total = sum(p.numel() for p in fresh_block.parameters())

print(f"Weights-only (assignment formula 4d^2 + 2*d*d_ff): {weights_only:,}")
print(f"  + MHA biases (4d):                               {mha_biases:,}")
print(f"  + FFN biases (d_ff + d):                          {ffn_biases:,}")
print(f"  + LayerNorm gamma/beta (2 x 2d):                  {ln_params:,}")
print(f"Derived grand total:                                {derived_total:,}")
print(f"Actual sum(p.numel()) on a real EncoderBlock:       {actual_total:,}")

assert weights_only == 196_608, f"expected 196,608 per the assignment, got {weights_only}"
assert derived_total == actual_total, \
    f"hand derivation ({derived_total}) must match sum(p.numel()) ({actual_total})"

print("Parameter-count reconciliation passed: 196,608 weights-only + 1,664 (biases+LN) = "
      f"{actual_total:,} total trainable params per block.")

Weights-only (assignment formula 4d^2 + 2*d*d_ff): 196,608
  + MHA biases (4d):                               512
  + FFN biases (d_ff + d):                          640
  + LayerNorm gamma/beta (2 x 2d):                  512
Derived grand total:                                198,272
Actual sum(p.numel()) on a real EncoderBlock:       198,272
Parameter-count reconciliation passed: 196,608 weights-only + 1,664 (biases+LN) = 198,272 total trainable params per block.


In [31]:
#  end-to-end TransformerEncoder stack sanity (4 layers, matches your d/heads/layers config):
# Matches the Configuration section of the README: d=128, heads=4, layers=4
encoder_stack = TransformerEncoder(d_model=128, n_heads=4, d_ff=512, n_layers=4, dropout=0.1)
total_stack_params = sum(p.numel() for p in encoder_stack.parameters())
print(f"4-layer encoder stack total trainable params: {total_stack_params:,}")

x = torch.randn(2, 10, 128)
out = encoder_stack(x)
assert out.shape == x.shape

# Causal variant works end-to-end too (needed for Task B later)
causal_out = encoder_stack(x, mask=causal_mask(10))
assert causal_out.shape == x.shape
print("TransformerEncoder stack forward pass (bidirectional and causal) passed.")

4-layer encoder stack total trainable params: 793,344
TransformerEncoder stack forward pass (bidirectional and causal) passed.


## Step 4 — HAR Encoder Classifier + Positional-Encoding Ablation (P2.3)
Reuses Problem 1's HAR data pipeline and generic training loop unchanged,
so the comparison against the best Problem 1 model (GRU, seed 999) isolates
architecture as the only variable. Ablates positional information:
none vs sinusoidal vs learned, 3 seeds each (9 runs total).

In [32]:
# Reuse Problem 1's data pipeline + generic training loop (no duplication).
PROBLEM1_MODULES_DIR = os.path.join(PROJECT_ROOT, "Problem1", "modules")
HAR_DATA_ROOT = os.path.join(PROJECT_ROOT, "Data", "har")
P2_CHECKPOINTS_DIR = os.path.join(PROJECT_ROOT, "Problem2", "checkpoints")

if PROBLEM1_MODULES_DIR not in sys.path:
    sys.path.insert(0, PROBLEM1_MODULES_DIR)

os.makedirs(P2_CHECKPOINTS_DIR, exist_ok=True)

print(f"Problem 1 modules dir: {PROBLEM1_MODULES_DIR}")
print(f"HAR data root: {HAR_DATA_ROOT}")
print(f"Problem 2 checkpoints dir: {P2_CHECKPOINTS_DIR}")

Problem 1 modules dir: /content/drive/MyDrive/TheRealDNNExp/Problem1/modules
HAR data root: /content/drive/MyDrive/TheRealDNNExp/Data/har
Problem 2 checkpoints dir: /content/drive/MyDrive/TheRealDNNExp/Problem2/checkpoints


In [33]:
import urllib.request
import zipfile

from load_har_datasets import (
    load_har, create_subject_split, compute_normalization_stats,
    normalize_data, HARDataset, build_dataloaders,
)
from problem1trng import train_model, evaluate, compute_metrics

HAR_ZIP_URL = "https://archive.ics.uci.edu/static/public/240/human+activity+recognition+using+smartphones.zip"


def ensure_har_dataset(data_root: str) -> str:
    """Return the folder containing activity_labels.txt, downloading/extracting if missing."""
    for root, _, files in os.walk(data_root):
        if "activity_labels.txt" in files:
            return root

    os.makedirs(data_root, exist_ok=True)
    outer_zip = os.path.join(data_root, "har.zip")
    print("Downloading UCI HAR dataset...")
    urllib.request.urlretrieve(HAR_ZIP_URL, outer_zip)

    with zipfile.ZipFile(outer_zip) as zf:
        zf.extractall(data_root)
    for root, _, files in os.walk(data_root):
        for name in files:
            if name.endswith(".zip"):
                with zipfile.ZipFile(os.path.join(root, name)) as zf:
                    zf.extractall(root)
    for root, _, files in os.walk(data_root):
        if "activity_labels.txt" in files:
            return root
    raise FileNotFoundError("Could not locate activity_labels.txt after extraction")


HAR_DATA_DIR = ensure_har_dataset(HAR_DATA_ROOT)

x_train_full, y_train_full, subjects_train = load_har(HAR_DATA_DIR, "train")  # (7352,128,9) (7352,) (7352,)
x_test, y_test, subjects_test = load_har(HAR_DATA_DIR, "test")               # (2947,128,9) (2947,) (2947,)
print("train:", x_train_full.shape, "test:", x_test.shape)

Finished stacking HAR signals for split 'train'
Finished loading HAR dataset for split 'train'
Finished stacking HAR signals for split 'test'
Finished loading HAR dataset for split 'test'
train: (7352, 128, 9) test: (2947, 128, 9)


In [34]:
# import results_io + assert-test for the dedup/upsert logic:
from results_io import upsert_result_row, read_all_rows

_test_csv_path = os.path.join(PROJECT_ROOT, "Results", "_results_io_selftest.csv")
if os.path.isfile(_test_csv_path):
    os.remove(_test_csv_path)

upsert_result_row(_test_csv_path, {"problem": "TEST", "model": "a", "seed": 1, "seq_len": 10, "test_acc": 0.5})
upsert_result_row(_test_csv_path, {"problem": "TEST", "model": "b", "seed": 1, "seq_len": 10, "test_acc": 0.9})
rows = read_all_rows(_test_csv_path)
assert len(rows) == 2, "two distinct keys must produce two rows"

# Re-writing the SAME key with a different value must REPLACE, not duplicate.
upsert_result_row(_test_csv_path, {"problem": "TEST", "model": "a", "seed": 1, "seq_len": 10, "test_acc": 0.99})
rows = read_all_rows(_test_csv_path)
assert len(rows) == 2, f"expected 2 rows after re-running same key, got {len(rows)} (dedup failed)"
updated = next(r for r in rows if r["model"] == "a")
assert float(updated["test_acc"]) == 0.99, "existing row must be updated in place, not appended"

os.remove(_test_csv_path)
print("results_io dedup/upsert tests passed.")

results_io dedup/upsert tests passed.


In [35]:
# import har_encoder + shape/param/CLS-wiring assert-test:
from har_encoder import HAREncoderClassifier

torch.manual_seed(0)
dummy_x = torch.randn(4, 128, 9)  # (B, window_len, n_channels)

model_sin = HAREncoderClassifier(pos_encoding="sinusoidal")
out = model_sin(dummy_x)
assert out.shape == (4, 6), f"expected (4, 6) logits, got {out.shape}"

# none/sinusoidal add 0 trainable params; learned adds (n_patches+1)*d_model.
params_none = sum(p.numel() for p in HAREncoderClassifier(pos_encoding="none").parameters())
params_sin = sum(p.numel() for p in HAREncoderClassifier(pos_encoding="sinusoidal").parameters())
params_learned = sum(p.numel() for p in HAREncoderClassifier(pos_encoding="learned").parameters())
assert params_none == params_sin, "sinusoidal PE must add 0 trainable params vs none"
assert params_learned - params_sin == 17 * 128, \
    f"learned PE should add exactly (n_patches+1)*d_model=2176 params, got {params_learned - params_sin}"

# CLS token must actually attend to the patches (bidirectional, no mask):
# perturbing ANY patch should change the CLS output (unlike the causal MHA test in Step 3).
model_sin.eval()
with torch.no_grad():
    out1 = model_sin(dummy_x)
    perturbed = dummy_x.clone()
    perturbed[:, 0, :] += 5.0  # perturb the FIRST time step only
    out2 = model_sin(perturbed)
assert not torch.allclose(out1, out2, atol=1e-4), "CLS output must depend on every patch (bidirectional)"

print(f"params: none/sinusoidal={params_sin:,}  learned={params_learned:,}  (+{params_learned - params_sin:,})")
print("HAREncoderClassifier shape, param-count, and CLS-wiring tests passed.")

params: none/sinusoidal=803,590  learned=805,766  (+2,176)
HAREncoderClassifier shape, param-count, and CLS-wiring tests passed.


In [36]:
# 9-run training loop (3 positional-encoding arms x 3 seeds):
from results_io import upsert_result_row

POS_MODES = ["none", "sinusoidal", "learned"]
SEEDS = [42, 123, 999]
EPOCHS = 30           # same budget as P1.3, for a fair comparison
BATCH_SIZE = 64       # same as P1.3
NUM_CLASSES = 6

RESULTS_CSV_PATH = os.path.join(PROJECT_ROOT, "Results", "results.csv")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

har_run_results = []

for pos_mode in POS_MODES:
    for seed in SEEDS:
        print(f"\n=== Training HAR encoder | pos={pos_mode} | seed={seed} ===")
        torch.manual_seed(seed)

        # Same subject-wise split logic/seed as P1.3, for an apples-to-apples split.
        train_mask, val_mask = create_subject_split(subjects_train, seed)
        x_tr, y_tr = x_train_full[train_mask], y_train_full[train_mask]
        x_va, y_va = x_train_full[val_mask], y_train_full[val_mask]

        mean, std = compute_normalization_stats(x_tr)
        x_tr_norm = normalize_data(x_tr, mean, std)
        x_va_norm = normalize_data(x_va, mean, std)
        x_te_norm = normalize_data(x_test, mean, std)

        train_ds = HARDataset(x_tr_norm, y_tr)
        val_ds = HARDataset(x_va_norm, y_va)
        test_ds = HARDataset(x_te_norm, y_test)
        train_loader, val_loader, test_loader = build_dataloaders(train_ds, val_ds, test_ds, BATCH_SIZE)

        model = HAREncoderClassifier(
            n_channels=9, window_len=128, patch_len=8,
            d_model=128, n_heads=4, d_ff=512, n_layers=4,
            n_classes=NUM_CLASSES, pos_encoding=pos_mode, dropout=0.1,
        ).to(device)

        sec_per_epoch = train_model(model, train_loader, val_loader, EPOCHS, lr=1e-3, device=device)

        y_true, y_pred = evaluate(model, test_loader, device)
        metrics = compute_metrics(y_true, y_pred)
        params = sum(p.numel() for p in model.parameters())

        checkpoint_path = os.path.join(P2_CHECKPOINTS_DIR, f"har_transformer_pos-{pos_mode}_seed{seed}.pt")
        torch.save(model.state_dict(), checkpoint_path)

        row = {
            "problem": "P2.3",
            "model": f"transformer_pos-{pos_mode}",
            "seed": seed,
            "seq_len": 128,
            "test_acc": metrics["test_acc"],
            "macro_f1": metrics["macro_f1"],
            "params": params,
            "sec_per_epoch": sec_per_epoch,
            "notes": f"HAR encoder classifier, patch_len=8, d=128, heads=4, layers=4, pos={pos_mode}",
        }
        upsert_result_row(RESULTS_CSV_PATH, row)
        har_run_results.append(row)

        print(f"  test_acc={metrics['test_acc']:.4f}  macro_f1={metrics['macro_f1']:.4f}  "
              f"params={params}  sec/epoch={sec_per_epoch:.2f}  checkpoint={checkpoint_path}")

print("\nAll 9 HAR encoder runs complete.")

Using device: cuda

=== Training HAR encoder | pos=none | seed=42 ===
Finished creating subject split with seed '42'
Finished computing normalization stats
Finished normalizing data with provided mean-[[[-6.5718323e-04 -3.0924269e-04 -3.7290764e-04 -1.9747333e-03
   -3.3446428e-04  9.3646644e-04  8.0351132e-01  3.3922240e-02
    8.7284394e-02]]] and std-[[[0.19556628 0.12406556 0.1058247  0.4125937  0.38717854 0.25981337
   0.41742373 0.39939222 0.34735426]]]
Finished normalizing data with provided mean-[[[-6.5718323e-04 -3.0924269e-04 -3.7290764e-04 -1.9747333e-03
   -3.3446428e-04  9.3646644e-04  8.0351132e-01  3.3922240e-02
    8.7284394e-02]]] and std-[[[0.19556628 0.12406556 0.1058247  0.4125937  0.38717854 0.25981337
   0.41742373 0.39939222 0.34735426]]]
Finished normalizing data with provided mean-[[[-6.5718323e-04 -3.0924269e-04 -3.7290764e-04 -1.9747333e-03
   -3.3446428e-04  9.3646644e-04  8.0351132e-01  3.3922240e-02
    8.7284394e-02]]] and std-[[[0.19556628 0.12406556 0.1

In [37]:
# comparison table vs best Problem 1 model (reads back from results.csv):
results_df = pd.read_csv(RESULTS_CSV_PATH)

p1_df = results_df[results_df["problem"] == "P1.3"]
best_p1_row = p1_df.loc[p1_df["macro_f1"].idxmax()]

p2_df = results_df[results_df["problem"] == "P2.3"].copy()
pos_summary = p2_df.groupby("model")[["test_acc", "macro_f1", "sec_per_epoch"]].mean().round(4)
pos_summary["params"] = p2_df.groupby("model")["params"].first()
print("Positional-encoding ablation (mean over 3 seeds):")
display(pos_summary)

best_p2_model = pos_summary["macro_f1"].idxmax()
best_p2_row = pos_summary.loc[best_p2_model]

comparison = pd.DataFrame({
    "best_P1_model (GRU)": [best_p1_row["test_acc"], best_p1_row["macro_f1"],
                             best_p1_row["params"], best_p1_row["sec_per_epoch"]],
    f"best_P2_model ({best_p2_model})": [best_p2_row["test_acc"], best_p2_row["macro_f1"],
                                          best_p2_row["params"], best_p2_row["sec_per_epoch"]],
}, index=["test_acc", "macro_f1", "params", "sec_per_epoch"])
print("\nBest-vs-best comparison:")
display(comparison)

Positional-encoding ablation (mean over 3 seeds):


,test_acc,macro_f1,sec_per_epoch,params
model,,,,
transformer_pos-learned,0.9077,0.9086,1.3577,805766
transformer_pos-none,0.9216,0.9211,1.3346,803590
transformer_pos-sinusoidal,0.9102,0.9096,1.3388,803590



Best-vs-best comparison:


,best_P1_model (GRU),best_P2_model (transformer_pos-none)
test_acc,0.922633,0.9216
macro_f1,0.922800,0.9211
params,14790.000000,803590.0000
sec_per_epoch,0.256866,1.3346


## Step 5 — Character-Level Causal Language Model on Tiny Shakespeare (P2.4)
Context 128, d=128, 4 heads, 4 layers, same TransformerEncoder/causal_mask
as earlier steps. Reports validation loss (nats) and bits-per-character,
then generates 500 characters at temperature 0.2, 0.8 and 1.5 with
top-k sampling.

In [38]:
# download + load data + build vocab:
from char_lm import (
    ensure_tiny_shakespeare, load_text, build_vocab, encode, decode,
    train_val_split, get_batch, CharTransformerLM, estimate_loss,
)

TS_DATA_DIR = os.path.join(PROJECT_ROOT, "Data", "tiny_shakespeare")
ts_text_path = ensure_tiny_shakespeare(TS_DATA_DIR)
ts_text = load_text(ts_text_path)

chars, stoi, itos = build_vocab(ts_text)
VOCAB_SIZE = len(chars)
print(f"Corpus length: {len(ts_text):,} characters")
print(f"Vocabulary size: {VOCAB_SIZE} distinct characters")

ts_data = encode(ts_text, stoi)                      # (len(ts_text),) long tensor
ts_train_data, ts_val_data = train_val_split(ts_data, val_fraction=0.1)
print(f"Train chars: {len(ts_train_data):,}  Val chars: {len(ts_val_data):,}")

Corpus length: 1,115,394 characters
Vocabulary size: 65 distinct characters
Train chars: 1,003,854  Val chars: 111,540


In [39]:
# assert tests for vocab/encode/decode/split
assert VOCAB_SIZE == 65, f"Tiny Shakespeare should have 65 distinct characters, got {VOCAB_SIZE}"

# Encode/decode must round-trip exactly
sample = ts_text[:200]
assert decode(encode(sample, stoi).tolist(), itos) == sample, "encode/decode must round-trip"

# Split must be contiguous and cover the whole corpus with no overlap/loss
assert len(ts_train_data) + len(ts_val_data) == len(ts_data)
assert torch.equal(ts_data[:len(ts_train_data)], ts_train_data)
assert torch.equal(ts_data[len(ts_train_data):], ts_val_data)

print("Vocab/encode/decode/split tests passed.")

Vocab/encode/decode/split tests passed.


In [40]:
# shape test:
torch.manual_seed(0)
CONTEXT_LEN = 128
xb_test, yb_test = get_batch(ts_train_data, CONTEXT_LEN, batch_size=4, device=torch.device("cpu"))
assert xb_test.shape == (4, CONTEXT_LEN)
assert yb_test.shape == (4, CONTEXT_LEN)

# target must be the input shifted by exactly one character, by construction
# (can't check this generically since get_batch samples random windows, but we
# can verify the *shapes and dtypes* required for F.cross_entropy)
assert xb_test.dtype == torch.long and yb_test.dtype == torch.long

print("get_batch shape/dtype tests passed.")

get_batch shape/dtype tests passed.


In [41]:
# model instantiation + shape/param/causal tests:
from encoder import TransformerEncoder  # already imported in Step 3, re-imported here for clarity

D_MODEL, N_HEADS, D_FF, N_LAYERS = 128, 4, 512, 4

char_model = CharTransformerLM(
    vocab_size=VOCAB_SIZE, d_model=D_MODEL, n_heads=N_HEADS,
    d_ff=D_FF, n_layers=N_LAYERS, max_len=CONTEXT_LEN, dropout=0.1,
)

dummy_idx = torch.randint(0, VOCAB_SIZE, (2, CONTEXT_LEN))
logits = char_model(dummy_idx)
assert logits.shape == (2, CONTEXT_LEN, VOCAB_SIZE), f"unexpected logits shape {logits.shape}"

# Causal check: changing the LAST token must not change logits at earlier positions
# (reuses the same perturbation-based test idea as the Step 3 MHA causal test)
char_model.eval()
with torch.no_grad():
    logits_1 = char_model(dummy_idx)
    perturbed_idx = dummy_idx.clone()
    perturbed_idx[:, -1] = (perturbed_idx[:, -1] + 1) % VOCAB_SIZE  # change only the last token
    logits_2 = char_model(perturbed_idx)
assert torch.allclose(logits_1[:, :-1, :], logits_2[:, :-1, :], atol=1e-5), \
    "causal LM: earlier positions must be unaffected by a later-token change"
char_model.train()

char_lm_params = sum(p.numel() for p in char_model.parameters())
print(f"CharTransformerLM params: {char_lm_params:,}")
print("Model shape + causal-independence tests passed.")

CharTransformerLM params: 810,049
Model shape + causal-independence tests passed.


In [42]:
# training loop
import math
import time

import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
char_model = char_model.to(device)
print("Using device:", device)

BATCH_SIZE_LM = 64
MAX_ITERS = 5000
EVAL_INTERVAL = 500
EVAL_ITERS = 100
LR = 3e-4
LM_SEED = 42

torch.manual_seed(LM_SEED)
optimizer = torch.optim.AdamW(char_model.parameters(), lr=LR)

block_times = []
val_loss_history = []
block_start = time.time()

for it in range(1, MAX_ITERS + 1):
    xb, yb = get_batch(ts_train_data, CONTEXT_LEN, BATCH_SIZE_LM, device)
    logits = char_model(xb)                                           # (B, T, vocab_size)
    loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), yb.reshape(-1))

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if it % EVAL_INTERVAL == 0 or it == MAX_ITERS:
        block_times.append(time.time() - block_start)
        block_start = time.time()
        val_loss = estimate_loss(char_model, ts_val_data, CONTEXT_LEN, BATCH_SIZE_LM, EVAL_ITERS, device)
        val_loss_history.append(val_loss)
        print(f"iter {it}/{MAX_ITERS} - train_loss={loss.item():.4f} - "
              f"val_loss={val_loss:.4f} nats ({val_loss / math.log(2):.4f} bits/char) - "
              f"{block_times[-1]:.1f}s/{EVAL_INTERVAL}steps")

sec_per_block = float(np.mean(block_times))
final_val_loss_nats = val_loss_history[-1]
final_bits_per_char = final_val_loss_nats / math.log(2)

print(f"\nFinal val loss: {final_val_loss_nats:.4f} nats "
      f"(uniform-guess baseline = ln(65) = {math.log(65):.4f} nats)")
print(f"Final bits per character: {final_bits_per_char:.4f}")

Using device: cuda
iter 500/5000 - train_loss=2.1918 - val_loss=2.1698 nats (3.1303 bits/char) - 15.6s/500steps
iter 1000/5000 - train_loss=1.9112 - val_loss=1.9581 nats (2.8249 bits/char) - 17.3s/500steps
iter 1500/5000 - train_loss=1.7748 - val_loss=1.8497 nats (2.6686 bits/char) - 17.5s/500steps
iter 2000/5000 - train_loss=1.7056 - val_loss=1.7739 nats (2.5591 bits/char) - 18.1s/500steps
iter 2500/5000 - train_loss=1.6238 - val_loss=1.7144 nats (2.4734 bits/char) - 18.0s/500steps
iter 3000/5000 - train_loss=1.5047 - val_loss=1.6828 nats (2.4278 bits/char) - 17.7s/500steps
iter 3500/5000 - train_loss=1.5579 - val_loss=1.6623 nats (2.3981 bits/char) - 17.7s/500steps
iter 4000/5000 - train_loss=1.4764 - val_loss=1.6380 nats (2.3632 bits/char) - 18.0s/500steps
iter 4500/5000 - train_loss=1.4545 - val_loss=1.6114 nats (2.3248 bits/char) - 17.9s/500steps
iter 5000/5000 - train_loss=1.4433 - val_loss=1.6024 nats (2.3118 bits/char) - 17.8s/500steps

Final val loss: 1.6024 nats (uniform-gues

In [43]:
# Save results and checkpointing
from results_io import upsert_result_row

RESULTS_CSV_PATH = os.path.join(PROJECT_ROOT, "Results", "results.csv")
P2_CHECKPOINTS_DIR = os.path.join(PROJECT_ROOT, "Problem2", "checkpoints")
os.makedirs(P2_CHECKPOINTS_DIR, exist_ok=True)

checkpoint_path = os.path.join(P2_CHECKPOINTS_DIR, f"char_lm_seed{LM_SEED}.pt")
torch.save(char_model.state_dict(), checkpoint_path)

row = {
    "problem": "P2.4",
    "model": "transformer_char_lm",
    "seed": LM_SEED,
    "seq_len": CONTEXT_LEN,
    "params": char_lm_params,
    "sec_per_epoch": sec_per_block,  # seconds per 500-iteration block (no fixed-size "epoch" for this task)
    "val_loss_nats": final_val_loss_nats,
    "bits_per_char": final_bits_per_char,
    "notes": (
        f"Tiny Shakespeare char LM, d=128 heads=4 layers=4 context=128, "
        f"max_iters={MAX_ITERS}, eval_interval={EVAL_INTERVAL} "
        f"(sec_per_epoch = sec per {EVAL_INTERVAL}-iteration block, not a full-corpus epoch)"
    ),
}
upsert_result_row(RESULTS_CSV_PATH, row)
print(f"Saved results row and checkpoint: {checkpoint_path}")

Saved results row and checkpoint: /content/drive/MyDrive/TheRealDNNExp/Problem2/checkpoints/char_lm_seed42.pt


In [44]:
# generation at 3 temp and generation of sample test data
TOP_K = 40  # not specified by the assignment; chosen as a standard top-k value
TEMPERATURES = [0.2, 0.8, 1.5]
GEN_LENGTH = 500

seed_context = torch.tensor([[stoi["\n"]]], dtype=torch.long, device=device)  # start from a single newline

generated_samples_path = os.path.join(PROJECT_ROOT, "Results", "generated_samples.txt")
samples_text = []

for temp in TEMPERATURES:
    torch.manual_seed(LM_SEED)
    out_idx = char_model.generate(seed_context, max_new_tokens=GEN_LENGTH, temperature=temp, top_k=TOP_K)
    generated_str = decode(out_idx[0].tolist(), itos)
    header = f"=== Temperature {temp} (top_k={TOP_K}) ==="
    samples_text.append(f"{header}\n{generated_str}\n")
    print(f"\n{header}\n{generated_str[:200]}...\n")  # preview first 200 chars in notebook output

with open(generated_samples_path, "w", encoding="utf-8") as f:
    f.write("\n".join(samples_text))

print(f"\nSaved 3 labeled 500-char samples to {generated_samples_path}")


=== Temperature 0.2 (top_k=40) ===



AUTOLYCUS:
I do not the see the son the bear of the proclaim
And the the state of the prove of the counters.

LUCIO:
I was the prove of the man of the shall be so much
And the state of the proclaim...


=== Temperature 0.8 (top_k=40) ===

I consent to the news the red that down,
And so, and the common well, will not fear.

GREPET:
Well, come is a sick follower, to it thy brother's.

HENRY BOLINGBROKE:
O, and Henry, he is well were him...


=== Temperature 1.5 (top_k=40) ===

JULET:
Hence Did, Tybalt, muddital.
The but consul yet me, and by lap will.
If That revells well. Gocest;, be MicRENCE:
Hean, you'ld storow.

CORIOLANUS:
Mask prome my hand-He.

VALINIUS:
O brunkshic...


Saved 3 labeled 500-char samples to /content/drive/MyDrive/TheRealDNNExp/Results/generated_samples.txt


## Step 6 — Analysis and Ablations (P2.5)
(a) Attention heatmaps for two heads on one sentence, with interpretation.
(b) Heads = 1, 4, 8 at fixed d=128.
(c) Remove the causal mask; compare training loss and generation.
(d) Time and memory at context length 64/128/256/512, related to O(n^2 * d).

In [45]:
# Attention Heatmaps
from analysis import get_two_head_attention, attention_distance, plot_attention_heatmaps

HEATMAP_SENTENCE = "To be, or not to be: that is the question"  # must use only trained-vocab chars

weights_2head, tokens = get_two_head_attention(
    char_model, HEATMAP_SENTENCE, stoi, device, layer_idx=0, head_indices=(0, 1)
)
assert weights_2head.shape == (2, len(tokens), len(tokens))
assert torch.allclose(weights_2head.sum(dim=-1), torch.ones(2, len(tokens)), atol=1e-4), \
    "each row of each head's attention weights must sum to 1"

HEATMAP_PATH = os.path.join(FIGURES_DIR, "attention_heatmaps", "two_heads_sentence.png")
plot_attention_heatmaps(weights_2head, tokens, head_indices=(0, 1), save_path=HEATMAP_PATH)

dist_0 = attention_distance(weights_2head[0])
dist_1 = attention_distance(weights_2head[1])
print(f"Head 0 mean attention distance |i-j|: {dist_0:.2f}")
print(f"Head 1 mean attention distance |i-j|: {dist_1:.2f}")
print("(A small distance = locally-focused head; a large distance = long-range/global head.)")

Saved attention heatmaps to /content/drive/MyDrive/TheRealDNNExp/Problem2/figures/attention_heatmaps/two_heads_sentence.png
Head 0 mean attention distance |i-j|: 2.39
Head 1 mean attention distance |i-j|: 2.20
(A small distance = locally-focused head; a large distance = long-range/global head.)


In [46]:
# Heads = 1, 4, 8 ablation (fixed d as 128)
from char_lm import train_char_lm
from analysis import plot_heads_ablation
from results_io import upsert_result_row

HEADS_ABLATION_LIST = [1, 4, 8]
ABLATION_MAX_ITERS = 2000      # reduced budget vs the main 5000-iter P2.4 run -- fair, controlled comparison
ABLATION_EVAL_INTERVAL = 500
ABLATION_EVAL_ITERS = 50

heads_val_losses = []
for n_heads_ablate in HEADS_ABLATION_LIST:
    print(f"\n=== Training char LM | n_heads={n_heads_ablate} (ablation) ===")
    torch.manual_seed(42)
    ablate_model = CharTransformerLM(
        vocab_size=VOCAB_SIZE, d_model=128, n_heads=n_heads_ablate,
        d_ff=512, n_layers=4, max_len=CONTEXT_LEN, dropout=0.1,
    ).to(device)

    result = train_char_lm(
        ablate_model, ts_train_data, ts_val_data, CONTEXT_LEN, BATCH_SIZE_LM,
        ABLATION_MAX_ITERS, ABLATION_EVAL_INTERVAL, ABLATION_EVAL_ITERS, LR, device, use_mask=True,
    )
    heads_val_losses.append(result["final_val_loss"])

    row = {
        "problem": "P2.5b",
        "model": f"transformer_heads{n_heads_ablate}",
        "seed": 42,
        "seq_len": CONTEXT_LEN,
        "params": sum(p.numel() for p in ablate_model.parameters()),
        "sec_per_epoch": result["sec_per_block"],
        "val_loss_nats": result["final_val_loss"],
        "bits_per_char": result["final_val_loss"] / math.log(2),
        "notes": f"Heads ablation, d=128 fixed, n_heads={n_heads_ablate}, max_iters={ABLATION_MAX_ITERS}",
    }
    upsert_result_row(RESULTS_CSV_PATH, row)
    print(f"  n_heads={n_heads_ablate}: final val_loss={result['final_val_loss']:.4f} nats")

HEADS_ABLATION_PATH = os.path.join(FIGURES_DIR, "heads_ablation.png")
plot_heads_ablation(HEADS_ABLATION_LIST, heads_val_losses, HEADS_ABLATION_PATH)

assert len(heads_val_losses) == 3
print("Heads ablation (1/4/8) complete.")


=== Training char LM | n_heads=1 (ablation) ===
  iter 500/2000 - train_loss=2.1118 - val_loss=2.1474 nats - 13.6s/500steps
  iter 1000/2000 - train_loss=1.8817 - val_loss=1.9349 nats - 14.3s/500steps
  iter 1500/2000 - train_loss=1.7559 - val_loss=1.8230 nats - 14.6s/500steps
  iter 2000/2000 - train_loss=1.6608 - val_loss=1.7592 nats - 13.7s/500steps
  n_heads=1: final val_loss=1.7592 nats

=== Training char LM | n_heads=4 (ablation) ===
  iter 500/2000 - train_loss=2.1350 - val_loss=2.1745 nats - 16.3s/500steps
  iter 1000/2000 - train_loss=1.8986 - val_loss=1.9644 nats - 17.1s/500steps
  iter 1500/2000 - train_loss=1.7729 - val_loss=1.8359 nats - 17.4s/500steps
  iter 2000/2000 - train_loss=1.6574 - val_loss=1.7638 nats - 17.3s/500steps
  n_heads=4: final val_loss=1.7638 nats

=== Training char LM | n_heads=8 (ablation) ===
  iter 500/2000 - train_loss=2.1639 - val_loss=2.1966 nats - 20.2s/500steps
  iter 1000/2000 - train_loss=1.9192 - val_loss=1.9771 nats - 21.3s/500steps
  iter

In [47]:
# removal of causal mask
from analysis import plot_train_loss_comparison

NOMASK_MAX_ITERS = ABLATION_MAX_ITERS  # same budget as 6b for a fair, controlled comparison

print("=== Training causal (masked) baseline for comparison ===")
torch.manual_seed(42)
causal_baseline_model = CharTransformerLM(
    vocab_size=VOCAB_SIZE, d_model=128, n_heads=4, d_ff=512, n_layers=4,
    max_len=CONTEXT_LEN, dropout=0.1,
).to(device)
causal_result = train_char_lm(
    causal_baseline_model, ts_train_data, ts_val_data, CONTEXT_LEN, BATCH_SIZE_LM,
    NOMASK_MAX_ITERS, ABLATION_EVAL_INTERVAL, ABLATION_EVAL_ITERS, LR, device, use_mask=True,
)

print("\n=== Training NO-MASK ablation (bidirectional self-attention) ===")
torch.manual_seed(42)
no_mask_model = CharTransformerLM(
    vocab_size=VOCAB_SIZE, d_model=128, n_heads=4, d_ff=512, n_layers=4,
    max_len=CONTEXT_LEN, dropout=0.1,
).to(device)
no_mask_result = train_char_lm(
    no_mask_model, ts_train_data, ts_val_data, CONTEXT_LEN, BATCH_SIZE_LM,
    NOMASK_MAX_ITERS, ABLATION_EVAL_INTERVAL, ABLATION_EVAL_ITERS, LR, device, use_mask=False,
)

# Expected finding: without the mask, each position can attend to the NEXT
# input token directly (x[t+1], which equals the target y[t]) -- a trivial
# shortcut -- so training/val loss collapses far below the causal baseline.
print(f"\nCausal baseline final val_loss:  {causal_result['final_val_loss']:.4f} nats")
print(f"No-mask ablation final val_loss: {no_mask_result['final_val_loss']:.4f} nats")
assert no_mask_result["final_val_loss"] < causal_result["final_val_loss"], \
    "no-mask should achieve a lower (artificially easy) loss via the future-token shortcut"

NOMASK_PLOT_PATH = os.path.join(FIGURES_DIR, "causal_vs_nomask_train_loss.png")
plot_train_loss_comparison(
    causal_result["train_loss_history"], no_mask_result["train_loss_history"], NOMASK_PLOT_PATH
)

row = {
    "problem": "P2.5c", "model": "transformer_no_mask", "seed": 42, "seq_len": CONTEXT_LEN,
    "params": sum(p.numel() for p in no_mask_model.parameters()),
    "sec_per_epoch": no_mask_result["sec_per_block"],
    "val_loss_nats": no_mask_result["final_val_loss"],
    "bits_per_char": no_mask_result["final_val_loss"] / math.log(2),
    "notes": f"No-mask ablation, max_iters={NOMASK_MAX_ITERS}; loss looks great but generation should fail",
}
upsert_result_row(RESULTS_CSV_PATH, row)

=== Training causal (masked) baseline for comparison ===
  iter 500/2000 - train_loss=2.1350 - val_loss=2.1745 nats - 16.5s/500steps
  iter 1000/2000 - train_loss=1.8986 - val_loss=1.9644 nats - 17.3s/500steps
  iter 1500/2000 - train_loss=1.7729 - val_loss=1.8359 nats - 17.4s/500steps
  iter 2000/2000 - train_loss=1.6574 - val_loss=1.7638 nats - 17.4s/500steps

=== Training NO-MASK ablation (bidirectional self-attention) ===
  iter 500/2000 - train_loss=0.1086 - val_loss=0.0844 nats - 16.0s/500steps
  iter 1000/2000 - train_loss=0.0276 - val_loss=0.0221 nats - 16.5s/500steps
  iter 1500/2000 - train_loss=0.0190 - val_loss=0.0202 nats - 16.5s/500steps
  iter 2000/2000 - train_loss=0.0201 - val_loss=0.0185 nats - 16.4s/500steps

Causal baseline final val_loss:  1.7638 nats
No-mask ablation final val_loss: 0.0185 nats
Saved causal-vs-no-mask train loss plot to /content/drive/MyDrive/TheRealDNNExp/Problem2/figures/causal_vs_nomask_train_loss.png


In [48]:
# validating whether generation is failed as no mask model cannot generate since at inference there is no future token to cheat with
torch.manual_seed(0)
seed_ctx = torch.tensor([[stoi["\n"]]], dtype=torch.long, device=device)

print("=== Causal baseline generation (should look like text) ===")
causal_gen = decode(
    causal_baseline_model.generate(seed_ctx, max_new_tokens=200, temperature=0.8, top_k=40)[0].tolist(), itos
)
print(causal_gen)

print("\n=== No-mask ablation generation (expected: degenerate/garbage) ===")
no_mask_gen = decode(
    no_mask_model.generate(seed_ctx, max_new_tokens=200, temperature=0.8, top_k=40)[0].tolist(), itos
)
print(no_mask_gen)

# Saved separately from the required generated_samples.txt (that file is reserved
# for the 3 temperature samples from the main P2.4 model per the submission spec).
demo_path = os.path.join(PROJECT_ROOT, "Results", "no_mask_generation_demo.txt")
with open(demo_path, "w", encoding="utf-8") as f:
    f.write(f"=== Causal baseline ===\n{causal_gen}\n\n=== No-mask ablation ===\n{no_mask_gen}\n")
print(f"\nSaved comparison to {demo_path}")

=== Causal baseline generation (should look like text) ===

And your heave and in obsubine.

Second Murderer:
Now me sorrow thou to that for stervoul the branishe.

PETRLANA:
Come, you thou there's sile, and feverious barn
Noth than the pumpos of war, night an

=== No-mask ablation generation (expected: degenerate/garbage) ===

OOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOIOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOO

Saved comparison to /content/drive/MyDrive/TheRealDNNExp/Results/no_mask_generation_demo.txt


In [49]:
# Comparison between Time/memory and context length
from analysis import measure_time_memory, plot_time_memory_vs_context

CONTEXT_LENGTHS_TO_TEST = [64, 128, 256, 512]
TIMING_BATCH_SIZE = 16

timing_model = CharTransformerLM(
    vocab_size=VOCAB_SIZE, d_model=128, n_heads=4, d_ff=512, n_layers=4,
    max_len=max(CONTEXT_LENGTHS_TO_TEST), dropout=0.1,
).to(device)

timing_results = measure_time_memory(
    timing_model, CONTEXT_LENGTHS_TO_TEST, TIMING_BATCH_SIZE, VOCAB_SIZE, device, n_repeats=5
)

assert set(timing_results.keys()) == set(CONTEXT_LENGTHS_TO_TEST)
assert timing_results[512]["time_sec"] > timing_results[64]["time_sec"], \
    "time per step must grow with context length"

TIME_PLOT_PATH = os.path.join(FIGURES_DIR, "context_length_time.png")
MEM_PLOT_PATH = os.path.join(FIGURES_DIR, "context_length_memory.png")
plot_time_memory_vs_context(timing_results, TIME_PLOT_PATH, MEM_PLOT_PATH)

for T in CONTEXT_LENGTHS_TO_TEST:
    row = {
        "problem": "P2.5d", "model": "transformer_timing", "seed": "", "seq_len": T,
        "sec_per_epoch": timing_results[T]["time_sec"],
        "notes": f"peak_mem_MB={timing_results[T]['memory_mb']:.1f}",
    }
    upsert_result_row(RESULTS_CSV_PATH, row)

print("Context-length timing/memory profiling complete.")

  T=  64  time/step=16.1ms  peak_mem=110.6MB
  T= 128  time/step=15.3ms  peak_mem=160.5MB
  T= 256  time/step=20.8ms  peak_mem=297.7MB
  T= 512  time/step=58.2ms  peak_mem=751.9MB
Saved time/memory-vs-context plots to /content/drive/MyDrive/TheRealDNNExp/Problem2/figures/context_length_time.png and /content/drive/MyDrive/TheRealDNNExp/Problem2/figures/context_length_memory.png
Context-length timing/memory profiling complete.
